# Cloud Capacity & Cost Analytics — Python Cleaning Notebook

This notebook is the Python proof behind my cloud capacity and cost analytics portfolio project.

The data is simulated for portfolio purposes. It does not contain real company data, customer data, or confidential cloud billing data.

The goal of this notebook is simple:

1. Create a small cloud usage dataset.
2. Clean the data.
3. Run basic data quality checks.
4. Export the cleaned dataset for SQL and dashboard analysis.
5. Create a small watch list for high-cost and low-utilization services.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries loaded successfully.")

## 1. Create a small simulated raw dataset

In a real company, this kind of data could come from cloud billing exports, usage logs, or cost center reports.

For this portfolio project, I simulate the data so that I can show the full workflow without using confidential information.


In [ ]:
raw_data = [
    ["2026-01", "Compute", "EU-Central-1", "Data Team", 5200, 800, 8900.50, 78],
    ["2026-01", "Storage", "eu-central-1", "Data Team", 300, 14000, 4200.00, 65],
    ["2026-01", "Database", "eu-west-1", "Business Apps", 2100, 7000, 6100.75, 72],
    ["2026-02", "Compute", "eu-central-1", "Data Team", 5500, 850, 9400.30, 81],
    ["2026-02", "Storage", "EU Central 1", "Data Team", 320, 15100, 4550.00, 67],
    ["2026-02", "Database", "eu-west-1", "Business Apps", 2200, 7300, 6400.25, 74],
    ["2026-03", "Compute", "eu-central-1", "Data Team", 5900, 900, 10150.10, 83],
    ["2026-03", "Storage", "eu-central-1", "Data Team", 330, 16000, 4700.00, 66],
    ["2026-03", "Database", "eu-west-1", "Business Apps", 2300, 7700, 6900.80, 75],
    ["2026-04", "Compute", "eu-central-1", "Data Team", 6200, 950, 12600.40, 36],
    ["2026-04", "Database", "eu-west-1", "Business Apps", 2400, 8000, 8300.60, 39],
    ["2026-04", "Analytics", "us-east-1", "Reporting", 1800, 2500, 3900.20, 70],
    ["2026-05", "Compute", "eu-central-1", "Data Team", 6500, 970, 13800.90, 35],
    ["2026-05", "Storage", "eu-central-1", "Data Team", 350, 18000, 5100.00, 69],
    ["2026-05", "Database", "eu-west-1", "Business Apps", 2500, 8500, 9100.10, 38],
    ["2026-06", "Compute", "eu-central-1", "Data Team", 6900, 1000, 14900.50, 37],
    ["2026-06", "Storage", "eu-central-1", "Data Team", 370, 19500, 5400.30, 71],
    ["2026-06", "Database", "eu-west-1", "Business Apps", 2600, 9000, 9700.90, 40],

    # Intentional data quality issues for cleaning proof
    ["2026-06", "Database", "eu-west-1", "Business Apps", 2600, 9000, 9700.90, 40],  # duplicate
    ["2026-06", "Backup", "eu-west-1", "Operations", 150, 12000, -500.00, 55],      # negative cost
    ["2026-06", "Network", None, "Connectivity", 900, 400, 2100.00, 62],            # missing region
    ["2026-06", "Analytics", "us-east-1", "Reporting", 1900, 2700, 4100.50, 135],   # invalid utilization
]

columns = [
    "billing_month",
    "service_name",
    "region_name",
    "team",
    "compute_hours",
    "storage_gb",
    "monthly_cost_eur",
    "utilization_pct",
]

raw_df = pd.DataFrame(raw_data, columns=columns)

print("Raw dataset shape:", raw_df.shape)
raw_df.head(10)

## 2. Clean the data

The cleaning step fixes the issues that could make the dashboard unreliable.

I check for:

- duplicate records
- missing region values
- inconsistent region names
- negative cost values
- utilization values outside the expected 0–100 range


In [ ]:
clean_df = raw_df.copy()

# Standardize text values
clean_df["service_name"] = clean_df["service_name"].str.strip().str.title()
clean_df["team"] = clean_df["team"].str.strip()

# Standardize region names
region_mapping = {
    "EU-Central-1": "eu-central-1",
    "EU Central 1": "eu-central-1",
    "eu-central-1": "eu-central-1",
    "eu-west-1": "eu-west-1",
    "us-east-1": "us-east-1",
}
clean_df["region_name"] = clean_df["region_name"].map(region_mapping)

# Remove rows where important fields are missing
clean_df = clean_df.dropna(subset=["billing_month", "service_name", "region_name", "team"])

# Remove duplicate service-month-region-team records
clean_df = clean_df.drop_duplicates(
    subset=["billing_month", "service_name", "region_name", "team"],
    keep="first"
)

# Remove records with negative or impossible cost
clean_df = clean_df[clean_df["monthly_cost_eur"] >= 0]

# Keep only realistic utilization values
clean_df = clean_df[
    (clean_df["utilization_pct"] >= 0) &
    (clean_df["utilization_pct"] <= 100)
]

# Convert billing month into a datetime format for analysis
clean_df["billing_month"] = pd.to_datetime(clean_df["billing_month"])

# Add simple IDs for SQL-style modeling
service_id_map = {name: f"S{i+1:03d}" for i, name in enumerate(sorted(clean_df["service_name"].unique()))}
region_id_map = {name: f"R{i+1:03d}" for i, name in enumerate(sorted(clean_df["region_name"].unique()))}
team_id_map = {name: f"C{i+1:03d}" for i, name in enumerate(sorted(clean_df["team"].unique()))}

clean_df["service_id"] = clean_df["service_name"].map(service_id_map)
clean_df["region_id"] = clean_df["region_name"].map(region_id_map)
clean_df["cost_center_id"] = clean_df["team"].map(team_id_map)

# Reorder columns for easier reading
clean_df = clean_df[
    [
        "billing_month",
        "service_id",
        "service_name",
        "region_id",
        "region_name",
        "cost_center_id",
        "team",
        "compute_hours",
        "storage_gb",
        "monthly_cost_eur",
        "utilization_pct",
    ]
]

print("Cleaned dataset shape:", clean_df.shape)
clean_df.head(10)

## 3. Data quality checks

A dashboard can look beautiful and still be wrong if the data behind it is weak.

That is why I include simple checks before exporting the cleaned data.


In [ ]:
quality_checks = {
    "missing_values": int(clean_df.isna().sum().sum()),
    "duplicate_records": int(clean_df.duplicated(subset=["billing_month", "service_name", "region_name", "team"]).sum()),
    "negative_cost_records": int((clean_df["monthly_cost_eur"] < 0).sum()),
    "invalid_utilization_records": int(((clean_df["utilization_pct"] < 0) | (clean_df["utilization_pct"] > 100)).sum()),
    "total_records_after_cleaning": int(len(clean_df)),
}

quality_checks_df = pd.DataFrame(
    list(quality_checks.items()),
    columns=["quality_check", "result"]
)

quality_checks_df

## 4. Export cleaned data

This cleaned CSV is the dataset I would use for SQL analysis and dashboard building.

The export path is written for the GitHub folder structure:

`data/cleaned/cloud_usage_cleaned.csv`


In [ ]:
# When this notebook is inside the python folder, this path saves the CSV into ../data/cleaned/
output_path = Path("../data/cleaned/cloud_usage_cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(output_path, index=False)

print(f"Cleaned data exported to: {output_path}")

## 5. Business summary

After cleaning, I create a simple summary.

This is the bridge between technical work and business understanding.


In [ ]:
summary_by_service = (
    clean_df
    .groupby("service_name", as_index=False)
    .agg(
        total_cost_eur=("monthly_cost_eur", "sum"),
        avg_utilization_pct=("utilization_pct", "mean"),
        total_compute_hours=("compute_hours", "sum"),
        total_storage_gb=("storage_gb", "sum"),
    )
    .sort_values("total_cost_eur", ascending=False)
)

summary_by_service["total_cost_eur"] = summary_by_service["total_cost_eur"].round(2)
summary_by_service["avg_utilization_pct"] = summary_by_service["avg_utilization_pct"].round(1)

summary_by_service

## 6. Cost optimization watch list

This query-style output shows services that may need a closer review.

In this example, I flag services where:

- total cost is above €5,000
- average utilization is below 40%

This does not automatically mean the service is waste. It means the service should be reviewed before the cost grows further.


In [ ]:
watch_list = summary_by_service[
    (summary_by_service["total_cost_eur"] > 5000) &
    (summary_by_service["avg_utilization_pct"] < 40)
].copy()

watch_list

## Final note

This notebook shows the cleaning logic behind the project.

The important point is not only that the data becomes cleaner. The important point is that cleaner data makes the later SQL queries, dashboard KPIs, and business recommendations more trustworthy.
